In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

# Project root: parent of the notebooks directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory exists:", DATA_DIR.exists())
print("scikit-learn:", sklearn.__version__)

print("\nFiles in data/:")
if DATA_DIR.exists():
    for path in sorted(DATA_DIR.iterdir()):
        print(f"{path.name:35} {path.stat().st_size:,} bytes")
else:
    print("Data directory not found.")

Project root: c:\Dev\Projects\planet-hunter
Data directory exists: True
scikit-learn: 1.9.1

Files in data/:
processed                           0 bytes
raw                                 4,096 bytes


In [2]:
for folder in [DATA_DIR / "raw", DATA_DIR / "processed"]:
    print(f"\n{folder.relative_to(PROJECT_ROOT)}/")

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(
                    f"  {path.relative_to(PROJECT_ROOT)} "
                    f"({path.stat().st_size:,} bytes)"
                )


data\raw/
  data\raw\sample_submission.csv (174,354 bytes)
  data\raw\test_cadences.csv (26,544 bytes)
  data\raw\test_centroid.npy (36,820,128 bytes)
  data\raw\test_flux.npy (36,820,128 bytes)
  data\raw\test_stars.csv (317,452 bytes)
  data\raw\train_cadences.csv (24,119 bytes)
  data\raw\train_centroid.npy (94,680,128 bytes)
  data\raw\train_flux.npy (94,680,128 bytes)
  data\raw\train_stars.csv (1,047,629 bytes)

data\processed/


In [3]:
# Dataset paths
RAW_DIR = DATA_DIR / "raw"

# Load metadata
train_stars = pd.read_csv(RAW_DIR / "train_stars.csv")
test_stars = pd.read_csv(RAW_DIR / "test_stars.csv")

train_cadences = pd.read_csv(RAW_DIR / "train_cadences.csv")
test_cadences = pd.read_csv(RAW_DIR / "test_cadences.csv")

# Memory-map the large arrays to avoid loading everything into RAM immediately
train_flux = np.load(
    RAW_DIR / "train_flux.npy", mmap_mode="r"
)
test_flux = np.load(
    RAW_DIR / "test_flux.npy", mmap_mode="r"
)

train_centroid = np.load(
    RAW_DIR / "train_centroid.npy", mmap_mode="r"
)
test_centroid = np.load(
    RAW_DIR / "test_centroid.npy", mmap_mode="r"
)

# Basic integrity checks
assert train_flux.shape[0] == len(train_stars)
assert test_flux.shape[0] == len(test_stars)
assert train_centroid.shape[0] == len(train_stars)
assert test_centroid.shape[0] == len(test_stars)

assert "label" in train_stars.columns
assert "label" not in test_stars.columns

print("Training flux:", train_flux.shape, train_flux.dtype)
print("Test flux:", test_flux.shape, test_flux.dtype)
print("Training centroid:", train_centroid.shape, train_centroid.dtype)
print("Test centroid:", test_centroid.shape, test_centroid.dtype)

print("\nTraining rows:", len(train_stars))
print("Test rows:", len(test_stars))
print("\nClass distribution:")
print(train_stars["label"].value_counts())
print("\nCadence quality counts:")
print(train_cadences["quality"].value_counts().sort_index())

Training flux: (18000, 1315) float32
Test flux: (7000, 1315) float32
Training centroid: (18000, 2, 1315) float16
Test centroid: (7000, 2, 1315) float16

Training rows: 18000
Test rows: 7000

Class distribution:
label
quiet_star           3624
starspots            2882
eclipsing_binary     2247
planet               2145
pulsating_star       1973
flare_star           1804
instrument_glitch    1721
background_binary    1604
Name: count, dtype: int64

Cadence quality counts:
quality
0    1221
1       8
2      33
4      53
Name: count, dtype: int64


In [4]:
# Keep only cadences marked as good quality
quality_mask = train_cadences["quality"].to_numpy() == 0

train_flux_q = np.asarray(train_flux[:, quality_mask], dtype=np.float32)
test_flux_q = np.asarray(test_flux[:, quality_mask], dtype=np.float32)

train_centroid_q = np.asarray(
    train_centroid[:, :, quality_mask], dtype=np.float32
)
test_centroid_q = np.asarray(
    test_centroid[:, :, quality_mask], dtype=np.float32
)

print("Good cadences:", quality_mask.sum())
print("Training flux:", train_flux_q.shape)
print("Test flux:", test_flux_q.shape)
print("Training centroid:", train_centroid_q.shape)
print("Test centroid:", test_centroid_q.shape)

Good cadences: 1221
Training flux: (18000, 1221)
Test flux: (7000, 1221)
Training centroid: (18000, 2, 1221)
Test centroid: (7000, 2, 1221)


In [5]:

FEATURE_DIR = PROJECT_ROOT / "src" / "features"

for path in sorted(FEATURE_DIR.glob("*.py")):
    print(f"\n--- {path.name} ---")
    print(path.read_text(encoding="utf-8")[:1200])


--- __init__.py ---


--- artifacts.py ---

import numpy as np
import pandas as pd


def extract_artifact_features(flux, batch=2000):
    """Extract robust sharp-outlier and jump features."""
    flux = np.asarray(flux, dtype=np.float32)
    n_stars = flux.shape[0]

    output = np.empty((n_stars, 3), dtype=np.float32)

    for start in range(0, n_stars, batch):
        end = min(start + batch, n_stars)
        values = flux[start:end]

        median = np.nanmedian(values, axis=1, keepdims=True)
        mad = np.nanmedian(
            np.abs(values - median), axis=1, keepdims=True
        )

        # Avoid division by zero for nearly constant curves.
        scale = np.maximum(1.4826 * mad, 1e-8)
        robust_z = np.abs((values - median) / scale)

        # Ignore missing samples in the outlier fraction.
        valid = np.isfinite(values)
        outlier = (robust_z > 8) & valid
        outlier_count = outlier.sum(axis=1)
        valid_count = valid.sum(axis=1)

        outlier_f

In [6]:
import ast

for path in sorted(FEATURE_DIR.glob("*.py")):
    if path.name == "__init__.py":
        continue

    print(f"\n{'=' * 70}\n{path.name}\n{'=' * 70}")

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(f"\nFunction: {ast.unparse(node.args)}")
            print(ast.get_source_segment(source, node))


artifacts.py

Function: flux, batch=2000
def extract_artifact_features(flux, batch=2000):
    """Extract robust sharp-outlier and jump features."""
    flux = np.asarray(flux, dtype=np.float32)
    n_stars = flux.shape[0]

    output = np.empty((n_stars, 3), dtype=np.float32)

    for start in range(0, n_stars, batch):
        end = min(start + batch, n_stars)
        values = flux[start:end]

        median = np.nanmedian(values, axis=1, keepdims=True)
        mad = np.nanmedian(
            np.abs(values - median), axis=1, keepdims=True
        )

        # Avoid division by zero for nearly constant curves.
        scale = np.maximum(1.4826 * mad, 1e-8)
        robust_z = np.abs((values - median) / scale)

        # Ignore missing samples in the outlier fraction.
        valid = np.isfinite(values)
        outlier = (robust_z > 8) & valid
        outlier_count = outlier.sum(axis=1)
        valid_count = valid.sum(axis=1)

        outlier_fraction = np.divide(
            outlier_cou

In [7]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis

sys.path.insert(0, str(PROJECT_ROOT))

from src.features.centroid import extract_centroid_features
from src.features.periodic import extract_periodic_features
from src.features.artifacts import extract_artifact_features
from src.features.dip import dip_features


def extract_base_features(flux, stars, batch=2000):
    """Extract the original 15 statistical and stellar features."""
    feature_names = [
        "std", "mad", "min", "max", "skew", "kurtosis",
        "p01", "p05", "p25", "p75", "p95", "p99",
        "tmag", "teff", "logg", "radius", "contamination",
    ]

    # There are 12 flux statistics and 5 stellar properties = 17.
    output = []

    for start in range(0, len(flux), batch):
        end = min(start + batch, len(flux))
        values = np.asarray(flux[start:end], dtype=np.float32)

        median = np.nanmedian(values, axis=1, keepdims=True)
        normalized = values / np.where(
            np.abs(median) < 1e-8, np.nan, median
        )

        stats = np.column_stack([
            np.nanstd(normalized, axis=1),
            np.nanmedian(
                np.abs(normalized - np.nanmedian(
                    normalized, axis=1, keepdims=True
                )), axis=1
            ),
            np.nanmin(normalized, axis=1),
            np.nanmax(normalized, axis=1),
            skew(normalized, axis=1, nan_policy="omit"),
            kurtosis(normalized, axis=1, nan_policy="omit"),
            *[
                np.nanpercentile(normalized, q, axis=1)
                for q in [1, 5, 25, 75, 95, 99]
            ],
        ])

        output.append(stats)

    flux_features = np.vstack(output)
    stellar_features = stars[
        ["tmag", "teff", "logg", "radius", "contamination"]
    ].to_numpy(dtype=np.float32)

    # The flux portion has 12 features, not 15.
    names = [
        "std", "mad", "min", "max", "skew", "kurtosis",
        "p01", "p05", "p25", "p75", "p95", "p99",
    ]

    return pd.DataFrame(
        np.column_stack([flux_features, stellar_features]),
        columns=names + [
            "tmag", "teff", "logg", "radius", "contamination"
        ],
    )


def extract_temporal_features(flux):
    """Extract six temporal features; the baseline uses three."""
    flux = np.asarray(flux, dtype=np.float32)
    median = np.nanmedian(flux, axis=1, keepdims=True)
    normalized = flux / np.where(np.abs(median) < 1e-8, np.nan, median)

    differences = np.diff(normalized, axis=1)
    diff_std = np.nanstd(differences, axis=1)
    diff_abs_median = np.nanmedian(np.abs(differences), axis=1)

    center = np.nanmedian(normalized, axis=1, keepdims=True)
    mad = np.nanmedian(np.abs(normalized - center), axis=1, keepdims=True)
    scale = np.maximum(1.4826 * mad, 1e-8)
    robust_z = (normalized - center) / scale

    positive_outlier_frac = np.nanmean(robust_z > 3, axis=1)
    negative_outlier_frac = np.nanmean(robust_z < -3, axis=1)

    x = np.arange(normalized.shape[1], dtype=np.float32)
    x_centered = x - x.mean()
    y_centered = normalized - np.nanmean(
        normalized, axis=1, keepdims=True
    )
    trend_slope = np.nansum(
        y_centered * x_centered, axis=1
    ) / np.maximum(np.sum(x_centered**2), 1e-8)

    left = normalized[:, :-1]
    right = normalized[:, 1:]
    left_centered = left - np.nanmean(left, axis=1, keepdims=True)
    right_centered = right - np.nanmean(right, axis=1, keepdims=True)
    denom = np.sqrt(
        np.nansum(left_centered**2, axis=1)
        * np.nansum(right_centered**2, axis=1)
    )
    lag1_corr = np.divide(
        np.nansum(left_centered * right_centered, axis=1),
        denom,
        out=np.zeros(len(flux), dtype=np.float32),
        where=denom > 1e-12,
    )

    return pd.DataFrame({
        "diff_std": diff_std,
        "diff_abs_median": diff_abs_median,
        "positive_outlier_frac": positive_outlier_frac,
        "negative_outlier_frac": negative_outlier_frac,
        "trend_slope": trend_slope,
        "lag1_corr": lag1_corr,
    })

In [8]:
# Inspect camera/CCD group sizes before detrending

print("Training stars per camera/CCD:")
print(
    train_stars.groupby(["camera", "ccd"])
    .size()
    .unstack(fill_value=0)
)

print("\nTest stars per camera/CCD:")
print(
    test_stars.groupby(["camera", "ccd"])
    .size()
    .unstack(fill_value=0)
)

print("\nTraining cadence time range:")
print(train_cadences["time_btjd"].agg(["min", "max"]))

print("\nTest cadence time range:")
print(test_cadences["time_btjd"].agg(["min", "max"]))

Training stars per camera/CCD:
ccd        1     2     3     4
camera                        
1       1150  1143  1145  1131
2       1146  1166  1178  1065
3       1095  1079  1133  1114
4       1082  1131  1100  1142

Test stars per camera/CCD:
ccd       1    2    3    4
camera                    
1       454  444  456  438
2       460  451  406  458
3       454  425  435  440
4       408  382  457  432

Training cadence time range:
min    2104.272
max    2131.647
Name: time_btjd, dtype: float64

Test cadence time range:
min    2131.877
max    2159.252
Name: time_btjd, dtype: float64


In [17]:
import numpy as np
import pandas as pd

# 1. Base features: 17
X_base = extract_base_features(train_flux_q, train_stars)
X_test_base = extract_base_features(test_flux_q, test_stars)

# 2. Temporal features: keep the same 3 used by our baseline
X_temporal_all = extract_temporal_features(train_flux_q)
X_test_temporal_all = extract_temporal_features(test_flux_q)

temporal_cols = ["trend_slope", "diff_abs_median", "diff_std"]

X_temporal = X_temporal_all[temporal_cols]
X_test_temporal = X_test_temporal_all[temporal_cols]

# 3. Centroid features: 8
X_centroid = extract_centroid_features(train_centroid_q)
X_test_centroid = extract_centroid_features(test_centroid_q)

# 4. Periodic features: 5
X_periodic = extract_periodic_features(train_flux_q)
X_test_periodic = extract_periodic_features(test_flux_q)

# 5. Artifact features: 3
X_artifact = extract_artifact_features(train_flux_q)
X_test_artifact = extract_artifact_features(test_flux_q)

# 6. Dip features: 4
X_dip = dip_features(train_flux_q)
X_test_dip = dip_features(test_flux_q)

# Combine in the same feature groups as the baseline
feature_frames = [
    X_base,
    X_temporal,
    X_centroid,
    X_periodic,
    X_artifact,
    X_dip,
]

test_feature_frames = [
    X_test_base,
    X_test_temporal,
    X_test_centroid,
    X_test_periodic,
    X_test_artifact,
    X_test_dip,
]

X_quality_dip = pd.concat(feature_frames, axis=1)
X_test_quality_dip = pd.concat(test_feature_frames, axis=1)

# Align test columns to training columns
X_test_quality_dip = X_test_quality_dip[X_quality_dip.columns]

print("Train features:", X_quality_dip.shape)
print("Test features:", X_test_quality_dip.shape)
print("Matching columns:",
      X_quality_dip.columns.equals(X_test_quality_dip.columns))
print("All training features finite:",
      np.isfinite(X_quality_dip.to_numpy()).all())
print("All test features finite:",
      np.isfinite(X_test_quality_dip.to_numpy()).all())

Train features: (18000, 40)
Test features: (7000, 40)
Matching columns: True
All training features finite: True
All test features finite: True


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

# Use the labels from the training metadata
y = train_stars["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X_quality_dip,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

baseline_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0,
)

baseline_model.fit(X_train, y_train)
val_pred = baseline_model.predict(X_val)

baseline_f1 = f1_score(
    y_val, val_pred, average="macro"
)

print(f"Baseline validation macro-F1: {baseline_f1:.4f}")
print()
print(classification_report(y_val, val_pred, digits=3))

Baseline validation macro-F1: 0.5308

                   precision    recall  f1-score   support

background_binary      0.459     0.467     0.463       321
 eclipsing_binary      0.825     0.579     0.681       449
       flare_star      0.544     0.499     0.520       361
instrument_glitch      0.650     0.599     0.623       344
           planet      0.411     0.382     0.396       429
   pulsating_star      0.604     0.638     0.621       395
       quiet_star      0.475     0.648     0.548       725
        starspots      0.422     0.370     0.394       576

         accuracy                          0.526      3600
        macro avg      0.549     0.523     0.531      3600
     weighted avg      0.539     0.526     0.527      3600



In [19]:
from sklearn.base import clone
from sklearn.metrics import f1_score

# Identify the periodic feature columns
periodic_cols = [
    col for col in X_quality_dip.columns
    if "autocorr" in col.lower() or "periodic" in col.lower()
]

print("Periodic columns found:", periodic_cols)

# Baseline score from the model already trained
print(f"All 40 features: {baseline_f1:.4f}")

# Remove only periodic features
X_no_periodic = X_quality_dip.drop(columns=periodic_cols)

X_train_np = X_no_periodic.loc[X_train.index]
X_val_np = X_no_periodic.loc[X_val.index]

model_no_periodic = clone(baseline_model)
model_no_periodic.fit(X_train_np, y_train)

pred_no_periodic = model_no_periodic.predict(X_val_np)
score_no_periodic = f1_score(
    y_val, pred_no_periodic, average="macro"
)

print(f"Without periodic features: {score_no_periodic:.4f}")
print(f"Difference: {score_no_periodic - baseline_f1:+.4f}")

Periodic columns found: ['autocorr_lag_5', 'autocorr_lag_10', 'autocorr_lag_20', 'autocorr_lag_50', 'autocorr_lag_100']
All 40 features: 0.5308
Without periodic features: 0.5121
Difference: -0.0188


In [20]:


metadata_cols = [
    "tmag", "teff", "logg", "radius", "contamination"
]

print("Metadata columns found:",
      [c for c in metadata_cols if c in X_quality_dip.columns])

# Keep all signal-derived features, remove only stellar metadata
cols_to_remove = [
    c for c in metadata_cols if c in X_quality_dip.columns
]

X_no_metadata = X_quality_dip.drop(columns=cols_to_remove)

X_train_nm = X_no_metadata.loc[X_train.index]
X_val_nm = X_no_metadata.loc[X_val.index]

model_no_metadata = clone(baseline_model)
model_no_metadata.fit(X_train_nm, y_train)

pred_no_metadata = model_no_metadata.predict(X_val_nm)

score_no_metadata = f1_score(
    y_val, pred_no_metadata, average="macro"
)

print(f"All 40 features:       {baseline_f1:.4f}")
print(f"Without metadata:      {score_no_metadata:.4f}")
print(f"Difference:            {score_no_metadata - baseline_f1:+.4f}")

Metadata columns found: ['tmag', 'teff', 'logg', 'radius', 'contamination']
All 40 features:       0.5308
Without metadata:      0.4976
Difference:            -0.0332


In [22]:
# Normalize each light curve by its median, with safe handling of zero medians.
# Use a relative flux representation: (flux / median) - 1.

def normalize_flux_per_star(flux):
    flux = np.asarray(flux, dtype=np.float32)

    med = np.median(flux, axis=1, keepdims=True)
    safe_med = np.where(np.abs(med) < 1e-8, 1.0, med)

    return (flux / safe_med) - 1.0


train_flux_norm = normalize_flux_per_star(train_flux_q)
test_flux_norm = normalize_flux_per_star(test_flux_q)

print("Normalized train:", train_flux_norm.shape)
print("Normalized test:", test_flux_norm.shape)
print("Train finite:", np.isfinite(train_flux_norm).all())
print("Test finite:", np.isfinite(test_flux_norm).all())

Normalized train: (18000, 1221)
Normalized test: (7000, 1221)
Train finite: True
Test finite: True


In [23]:
# Extract features from normalized flux curves
X_base_norm = extract_base_features(train_flux_norm, train_stars)
X_test_base_norm = extract_base_features(test_flux_norm, test_stars)

X_temporal_norm_all = extract_temporal_features(train_flux_norm)
X_test_temporal_norm_all = extract_temporal_features(test_flux_norm)

temporal_cols = ["trend_slope", "diff_abs_median", "diff_std"]

X_temporal_norm = X_temporal_norm_all[temporal_cols]
X_test_temporal_norm = X_test_temporal_norm_all[temporal_cols]

X_periodic_norm = extract_periodic_features(train_flux_norm)
X_test_periodic_norm = extract_periodic_features(test_flux_norm)

X_artifact_norm = extract_artifact_features(train_flux_norm)
X_test_artifact_norm = extract_artifact_features(test_flux_norm)

X_dip_norm = dip_features(train_flux_norm)
X_test_dip_norm = dip_features(test_flux_norm)

# Retain the original centroid features
feature_frames_norm = [
    X_base_norm, X_temporal_norm, X_centroid,
    X_periodic_norm, X_artifact_norm, X_dip_norm
]

test_feature_frames_norm = [
    X_test_base_norm, X_test_temporal_norm, X_test_centroid,
    X_test_periodic_norm, X_test_artifact_norm, X_test_dip_norm
]

X_norm = pd.concat(feature_frames_norm, axis=1)
X_test_norm = pd.concat(test_feature_frames_norm, axis=1)

# Ensure the columns match exactly
X_test_norm = X_test_norm[X_norm.columns]

print("Train:", X_norm.shape)
print("Test:", X_test_norm.shape)
print("Columns match:", X_norm.columns.equals(X_test_norm.columns))
print("Train finite:", np.isfinite(X_norm.to_numpy()).all())
print("Test finite:", np.isfinite(X_test_norm.to_numpy()).all())

c:\Dev\Projects\planet-hunter\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Dev\Projects\planet-hunter\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
C:\Users\kasir\AppData\Local\Temp\ipykernel_14016\2068007221.py:41: RuntimeWarning: All-NaN slice encountered
  np.nanmin(normalized, axis=1),
C:\Users\kasir\AppData\Local\Temp\ipykernel_14016\2068007221.py:42: RuntimeWarning: All-NaN slice encountered
  np.nanmax(normalized, axis=1),
C:\Users\kasir\AppData\Local\Temp\ipykernel_14016\2068007221.py:43: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  skew(normalized, axis=1, nan_p

Train: (18000, 40)
Test: (7000, 40)
Columns match: True
Train finite: False
Test finite: False


In [24]:
def inspect_array(name, arr):
    arr = np.asarray(arr)
    print(f"\n{name}")
    print("Shape:", arr.shape)
    print("NaN values:", np.isnan(arr).sum())
    print("Infinite values:", np.isinf(arr).sum())
    print("Rows with any NaN:", np.isnan(arr).any(axis=1).sum())
    print("Rows entirely NaN:", np.isnan(arr).all(axis=1).sum())

inspect_array("Original training flux", train_flux_q)
inspect_array("Normalized training flux", train_flux_norm)
inspect_array("Original test flux", test_flux_q)
inspect_array("Normalized test flux", test_flux_norm)


Original training flux
Shape: (18000, 1221)
NaN values: 0
Infinite values: 0
Rows with any NaN: 0
Rows entirely NaN: 0

Normalized training flux
Shape: (18000, 1221)
NaN values: 0
Infinite values: 0
Rows with any NaN: 0
Rows entirely NaN: 0

Original test flux
Shape: (7000, 1221)
NaN values: 0
Infinite values: 0
Rows with any NaN: 0
Rows entirely NaN: 0

Normalized test flux
Shape: (7000, 1221)
NaN values: 0
Infinite values: 0
Rows with any NaN: 0
Rows entirely NaN: 0


In [25]:
feature_groups = {
    "base": X_base_norm,
    "temporal": X_temporal_norm,
    "centroid": X_centroid,
    "periodic": X_periodic_norm,
    "artifact": X_artifact_norm,
    "dip": X_dip_norm,
}

test_groups = {
    "base": X_test_base_norm,
    "temporal": X_test_temporal_norm,
    "centroid": X_test_centroid,
    "periodic": X_test_periodic_norm,
    "artifact": X_test_artifact_norm,
    "dip": X_test_dip_norm,
}

for name, frame in feature_groups.items():
    values = np.asarray(frame)
    print(
        f"{name:10} | shape={values.shape} "
        f"| NaNs={np.isnan(values).sum()} "
        f"| infs={np.isinf(values).sum()}"
    )

print("\nTest feature groups:")
for name, frame in test_groups.items():
    values = np.asarray(frame)
    print(
        f"{name:10} | shape={values.shape} "
        f"| NaNs={np.isnan(values).sum()} "
        f"| infs={np.isinf(values).sum()}"
    )

base       | shape=(18000, 17) | NaNs=216000 | infs=0
temporal   | shape=(18000, 3) | NaNs=36000 | infs=0
centroid   | shape=(18000, 8) | NaNs=0 | infs=0
periodic   | shape=(18000, 5) | NaNs=90000 | infs=0
artifact   | shape=(18000, 3) | NaNs=0 | infs=0
dip        | shape=(18000, 4) | NaNs=0 | infs=0

Test feature groups:
base       | shape=(7000, 17) | NaNs=84000 | infs=0
temporal   | shape=(7000, 3) | NaNs=14000 | infs=0
centroid   | shape=(7000, 8) | NaNs=0 | infs=0
periodic   | shape=(7000, 5) | NaNs=35000 | infs=0
artifact   | shape=(7000, 3) | NaNs=0 | infs=0
dip        | shape=(7000, 4) | NaNs=0 | infs=0


In [26]:
for name, frame in [
    ("base", X_base_norm),
    ("temporal", X_temporal_norm),
    ("periodic", X_periodic_norm),
]:
    nan_counts = frame.isna().sum()
    affected = nan_counts[nan_counts > 0]

    print(f"\n{name}:")
    print(affected.to_string())


base:
std         18000
mad         18000
min         18000
max         18000
skew        18000
kurtosis    18000
p01         18000
p05         18000
p25         18000
p75         18000
p95         18000
p99         18000

temporal:
diff_abs_median    18000
diff_std           18000

periodic:
autocorr_lag_5      18000
autocorr_lag_10     18000
autocorr_lag_20     18000
autocorr_lag_50     18000
autocorr_lag_100    18000


In [27]:
print("Raw feature matrix finite:",
      np.isfinite(X_quality_dip.to_numpy()).all())

print("Raw test matrix finite:",
      np.isfinite(X_test_quality_dip.to_numpy()).all())

print("Baseline validation macro-F1:", round(baseline_f1, 4))

Raw feature matrix finite: True
Raw test matrix finite: True
Baseline validation macro-F1: 0.5308


In [28]:


X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X_quality_dip,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y,
)

model_seed123 = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0,
)

model_seed123.fit(X_train_s, y_train_s)
pred_seed123 = model_seed123.predict(X_val_s)

score_seed123 = f1_score(
    y_val_s, pred_seed123, average="macro"
)

print(f"Seed 42 validation:  {baseline_f1:.4f}")
print(f"Seed 123 validation: {score_seed123:.4f}")
print(f"Difference:          {score_seed123 - baseline_f1:+.4f}")

Seed 42 validation:  0.5308
Seed 123 validation: 0.5199
Difference:          -0.0109


In [29]:


def fit_hgb(seed, X_train, y_train, X_val):
    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.06,
        class_weight="balanced",
        random_state=seed,
    )
    model.fit(X_train, y_train)
    return model.predict_proba(X_val), model.classes_

# Use the same splits already created above.
# Train three models with different model seeds for split 42.
probs_42 = []
for seed in [0, 21, 42]:
    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.06,
        class_weight="balanced",
        random_state=seed,
    )
    model.fit(X_train, y_train)
    probs_42.append(model.predict_proba(X_val))

# HistGradientBoostingClassifier uses the same sorted class order
# when trained on the same labels.
avg_probs_42 = np.mean(probs_42, axis=0)
ensemble_pred_42 = baseline_model.classes_[np.argmax(avg_probs_42, axis=1)]

ensemble_f1_42 = f1_score(
    y_val, ensemble_pred_42, average="macro"
)

print(f"Single model, seed 42: {baseline_f1:.4f}")
print(f"Three-model ensemble:  {ensemble_f1_42:.4f}")
print(f"Difference:            {ensemble_f1_42 - baseline_f1:+.4f}")

Single model, seed 42: 0.5308
Three-model ensemble:  0.5261
Difference:            -0.0047


In [30]:
import numpy as np
from scipy.ndimage import median_filter

def detrend_flux(flux, window=101):
    """
    Remove slow variations using a rolling-median trend.
    Each row represents one star's light curve.
    """
    flux = np.asarray(flux, dtype=np.float32)

    trend = median_filter(
        flux,
        size=(1, window),
        mode="reflect"
    )

    # Subtract the estimated trend while preserving each star's
    # approximate central flux level.
    detrended = flux - trend + np.median(
        flux, axis=1, keepdims=True
    )

    return detrended.astype(np.float32)


train_flux_detrended = detrend_flux(train_flux_q)
test_flux_detrended = detrend_flux(test_flux_q)

print("Train:", train_flux_detrended.shape)
print("Test: ", test_flux_detrended.shape)
print("Train finite:", np.isfinite(train_flux_detrended).all())
print("Test finite: ", np.isfinite(test_flux_detrended).all())

Train: (18000, 1221)
Test:  (7000, 1221)
Train finite: True
Test finite:  True


In [33]:
# Build features from detrended flux
# Keep centroid features and stellar metadata unchanged.

X_base_det = extract_base_features(train_flux_detrended, train_stars)
X_base_test_det = extract_base_features(test_flux_detrended, test_stars)

X_temp_det = extract_temporal_features(train_flux_detrended)
X_temp_test_det = extract_temporal_features(test_flux_detrended)

X_cent_det = extract_centroid_features(train_centroid_q)
X_cent_test_det = extract_centroid_features(test_centroid_q)

X_periodic_det = extract_periodic_features(train_flux_detrended)
X_periodic_test_det = extract_periodic_features(test_flux_detrended)

X_artifact_det = extract_artifact_features(train_flux_detrended)
X_artifact_test_det = extract_artifact_features(test_flux_detrended)

X_dip_det = dip_features(train_flux_detrended)
X_dip_test_det = dip_features(test_flux_detrended)

print("Feature extraction complete.")

c:\Dev\Projects\planet-hunter\src\features\dip.py:42: RuntimeWarning: All-NaN slice encountered
  deepest_dip = np.nanmax(-z, axis=1)
c:\Dev\Projects\planet-hunter\src\features\dip.py:47: RuntimeWarning: Mean of empty slice
  down_mean = np.nanmean(down, axis=1)
c:\Dev\Projects\planet-hunter\src\features\dip.py:48: RuntimeWarning: Mean of empty slice
  up_mean = np.nanmean(up, axis=1)


Feature extraction complete.


In [34]:
feature_sets = {
    "base": X_base_det,
    "temporal": X_temp_det,
    "centroid": X_cent_det,
    "periodic": X_periodic_det,
    "artifact": X_artifact_det,
    "dip": X_dip_det,
}

for name, features in feature_sets.items():
    arr = np.asarray(features)
    print(
        f"{name:10s} shape={arr.shape}, "
        f"NaNs={np.isnan(arr).sum()}, "
        f"Infs={np.isinf(arr).sum()}"
    )

base       shape=(18000, 17), NaNs=0, Infs=0
temporal   shape=(18000, 6), NaNs=0, Infs=0
centroid   shape=(18000, 8), NaNs=0, Infs=0
periodic   shape=(18000, 5), NaNs=0, Infs=0
artifact   shape=(18000, 3), NaNs=0, Infs=0
dip        shape=(18000, 4), NaNs=0, Infs=0


In [35]:
# Select the same temporal features used in the original baseline.
# Original temporal feature order:
# diff_std, diff_abs_median, positive_outlier_frac,
# negative_outlier_frac, trend_slope, lag1_corr

temporal_cols = [4, 1, 0]  # trend_slope, diff_abs_median, diff_std

X_detrended = np.concatenate(
    [
        np.asarray(X_base_det),
        np.asarray(X_temp_det)[:, temporal_cols],
        np.asarray(X_cent_det),
        np.asarray(X_periodic_det),
        np.asarray(X_artifact_det),
        np.asarray(X_dip_det),
    ],
    axis=1,
)

X_test_detrended = np.concatenate(
    [
        np.asarray(X_base_test_det),
        np.asarray(X_temp_test_det)[:, temporal_cols],
        np.asarray(X_cent_test_det),
        np.asarray(X_periodic_test_det),
        np.asarray(X_artifact_test_det),
        np.asarray(X_dip_test_det),
    ],
    axis=1,
)

print("Train shape:", X_detrended.shape)
print("Test shape:", X_test_detrended.shape)
print("Train finite:", np.isfinite(X_detrended).all())
print("Test finite:", np.isfinite(X_test_detrended).all())

Train shape: (18000, 40)
Test shape: (7000, 40)
Train finite: True
Test finite: True


In [36]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

# Use the exact same split indices as the baseline.
# X_train and X_val were created from X_quality_dip.
train_idx = X_train.index if hasattr(X_train, "index") else None

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

indices = np.arange(len(y))

idx_train, idx_val = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

model_detrended = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0,
)

model_detrended.fit(X_detrended[idx_train], y.iloc[idx_train])
pred_detrended = model_detrended.predict(X_detrended[idx_val])

detrended_f1 = f1_score(
    y.iloc[idx_val],
    pred_detrended,
    average="macro",
)

print(f"Original baseline: 0.5308")
print(f"Detrended model:   {detrended_f1:.4f}")
print(f"Difference:        {detrended_f1 - 0.5308:+.4f}")
print("\nClassification report:")
print(classification_report(y.iloc[idx_val], pred_detrended))

Original baseline: 0.5308
Detrended model:   0.5271
Difference:        -0.0037

Classification report:
                   precision    recall  f1-score   support

background_binary       0.52      0.57      0.55       321
 eclipsing_binary       0.82      0.62      0.70       449
       flare_star       0.57      0.54      0.55       361
instrument_glitch       0.50      0.46      0.48       344
           planet       0.45      0.37      0.40       429
   pulsating_star       0.62      0.66      0.64       395
       quiet_star       0.45      0.63      0.52       725
        starspots       0.41      0.33      0.37       576

         accuracy                           0.52      3600
        macro avg       0.54      0.52      0.53      3600
     weighted avg       0.53      0.52      0.52      3600



In [38]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

class_names = sorted(y.unique())

print(classification_report(
    y_val,
    val_pred,
    labels=class_names,
    zero_division=0
))

cm = confusion_matrix(
    y_val,
    val_pred,
    labels=class_names
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {c}" for c in class_names],
    columns=[f"Pred: {c}" for c in class_names]
)

display(cm_df)

                   precision    recall  f1-score   support

background_binary       0.46      0.47      0.46       321
 eclipsing_binary       0.83      0.58      0.68       449
       flare_star       0.54      0.50      0.52       361
instrument_glitch       0.65      0.60      0.62       344
           planet       0.41      0.38      0.40       429
   pulsating_star       0.60      0.64      0.62       395
       quiet_star       0.48      0.65      0.55       725
        starspots       0.42      0.37      0.39       576

         accuracy                           0.53      3600
        macro avg       0.55      0.52      0.53      3600
     weighted avg       0.54      0.53      0.53      3600



,Pred: background_binary,Pred: eclipsing_binary,Pred: flare_star,Pred: instrument_glitch,Pred: planet,Pred: pulsating_star,Pred: quiet_star,Pred: starspots
Actual: background_binary,150,38,9,10,37,13,35,29
Actual: eclipsing_binary,78,260,9,7,33,7,26,29
Actual: flare_star,6,2,180,10,11,13,87,52
Actual: instrument_glitch,14,2,7,206,34,23,30,28
Actual: planet,30,7,13,31,164,20,100,64
Actual: pulsating_star,9,2,3,9,20,252,68,32
Actual: quiet_star,21,2,52,23,49,50,470,58
Actual: starspots,19,2,58,21,51,39,173,213


In [39]:
import numpy as np
from sklearn.metrics import f1_score

# Use probabilities from the ORIGINAL baseline model.
val_probs = baseline_model.predict_proba(X_val)
classes = baseline_model.classes_

print("Baseline macro-F1:", f1_score(
    y_val, classes[np.argmax(val_probs, axis=1)],
    average="macro"
))

# Increase selected class probabilities modestly before argmax.
# All other class probabilities remain unchanged.
for target_class in ["planet", "starspots", "background_binary"]:
    for multiplier in [1.10, 1.20, 1.30, 1.40]:
        adjusted = val_probs.copy()
        class_idx = np.where(classes == target_class)[0][0]
        adjusted[:, class_idx] *= multiplier

        pred = classes[np.argmax(adjusted, axis=1)]
        score = f1_score(y_val, pred, average="macro")

        print(f"{target_class:20s} x{multiplier:.2f}: {score:.4f}")

Baseline macro-F1: 0.5308062120832306
planet               x1.10: 0.5289
planet               x1.20: 0.5288
planet               x1.30: 0.5266
planet               x1.40: 0.5265
starspots            x1.10: 0.5312
starspots            x1.20: 0.5346
starspots            x1.30: 0.5355
starspots            x1.40: 0.5361
background_binary    x1.10: 0.5303
background_binary    x1.20: 0.5310
background_binary    x1.30: 0.5298
background_binary    x1.40: 0.5297


In [40]:
from sklearn.metrics import f1_score, precision_recall_fscore_support
import numpy as np

star_idx = np.where(classes == "starspots")[0][0]

for multiplier in [1.25, 1.35, 1.40, 1.45, 1.50]:
    adjusted = val_probs.copy()
    adjusted[:, star_idx] *= multiplier

    pred = classes[np.argmax(adjusted, axis=1)]

    macro = f1_score(y_val, pred, average="macro")
    _, _, class_f1, _ = precision_recall_fscore_support(
        y_val, pred, labels=classes, zero_division=0
    )
    star_f1 = class_f1[star_idx]

    print(
        f"Starspots x{multiplier:.2f} | "
        f"Macro-F1: {macro:.4f} | "
        f"Starspots F1: {star_f1:.4f}"
    )

Starspots x1.25 | Macro-F1: 0.5350 | Starspots F1: 0.4188
Starspots x1.35 | Macro-F1: 0.5362 | Starspots F1: 0.4304
Starspots x1.40 | Macro-F1: 0.5361 | Starspots F1: 0.4387
Starspots x1.45 | Macro-F1: 0.5351 | Starspots F1: 0.4423
Starspots x1.50 | Macro-F1: 0.5349 | Starspots F1: 0.4440


In [41]:
# Generate test probabilities using the original 40-feature model.
test_probs = baseline_model.predict_proba(X_test_quality_dip)

classes = baseline_model.classes_
star_idx = np.where(classes == "starspots")[0][0]

# Apply the locally best starspots multiplier.
adjusted_test_probs = test_probs.copy()
adjusted_test_probs[:, star_idx] *= 1.35

test_predictions = classes[np.argmax(adjusted_test_probs, axis=1)]

print("Predictions:", len(test_predictions))
print(pd.Series(test_predictions).value_counts())

Predictions: 7000
instrument_glitch    2202
starspots            1304
pulsating_star        805
planet                683
flare_star            597
eclipsing_binary      548
background_binary     484
quiet_star            377
Name: count, dtype: int64


In [42]:
print("Model training rows:", baseline_model.n_iter_ if False else
      getattr(baseline_model, "n_features_in_", "unknown"))
print("Training matrix rows:", X_train.shape[0])
print("Full feature matrix rows:", X_quality_dip.shape[0])

Model training rows: 40
Training matrix rows: 14400
Full feature matrix rows: 18000


In [43]:
final_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0,
)

final_model.fit(X_quality_dip, y)

final_test_probs = final_model.predict_proba(X_test_quality_dip)
final_classes = final_model.classes_

star_idx = np.where(final_classes == "starspots")[0][0]
final_test_probs[:, star_idx] *= 1.35

final_predictions = final_classes[
    np.argmax(final_test_probs, axis=1)
]

print("Predictions:", len(final_predictions))
print(pd.Series(final_predictions).value_counts())

Predictions: 7000
instrument_glitch    2359
starspots            1227
pulsating_star        804
flare_star            628
planet                627
eclipsing_binary      561
background_binary     465
quiet_star            329
Name: count, dtype: int64


In [46]:
raw_dir = Path(r"C:\Dev\Projects\planet-hunter\data\raw")
sample = pd.read_csv(raw_dir / "sample_submission.csv")
test_stars = pd.read_csv(raw_dir / "test_stars.csv")

In [48]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path.cwd().parent
raw_dir = project_dir / "data" / "raw"

sample = pd.read_csv(raw_dir / "sample_submission.csv")
test_stars = pd.read_csv(raw_dir / "test_stars.csv")

assert len(final_predictions) == len(test_stars) == len(sample) == 7000
assert np.isin(final_predictions, final_classes).all()

submission_3 = sample.copy()
submission_3["label"] = final_predictions

output_path = Path.cwd() / "artifacts" / "submission_3_starspots.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
submission_3.to_csv(output_path, index=False)

check = pd.read_csv(output_path)

print("Saved to:", output_path.resolve())
print("Shape:", check.shape)
print("Columns:", check.columns.tolist())
print("Missing values:", check.isna().sum().sum())
print("\nPrediction distribution:")
print(check["label"].value_counts())

Saved to: C:\Dev\Projects\planet-hunter\notebooks\artifacts\submission_3_starspots.csv
Shape: (7000, 2)
Columns: ['id', 'label']
Missing values: 0

Prediction distribution:
label
instrument_glitch    2359
starspots            1227
pulsating_star        804
flare_star            628
planet                627
eclipsing_binary      561
background_binary     465
quiet_star            329
Name: count, dtype: int64


In [49]:
required = [
    "X_quality_dip",
    "X_test_quality_dip",
    "y",
    "baseline_model",
]

for name in required:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")

if all(name in globals() for name in required):
    print("\nTrain features:", X_quality_dip.shape)
    print("Test features:", X_test_quality_dip.shape)
    print("Training labels:", y.shape)
    print("Baseline classes:", baseline_model.classes_)

X_quality_dip: available
X_test_quality_dip: available
y: available
baseline_model: available

Train features: (18000, 40)
Test features: (7000, 40)
Training labels: (18000,)
Baseline classes: ['background_binary' 'eclipsing_binary' 'flare_star' 'instrument_glitch'
 'planet' 'pulsating_star' 'quiet_star' 'starspots']


In [50]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, roc_auc_score

# Use the same validation split as our baseline experiment.
X_train, X_val, y_train, y_val = train_test_split(
    X_quality_dip,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train a classifier to distinguish training-sector rows from test-sector rows.
X_domain = np.vstack([X_quality_dip, X_test_quality_dip])
domain_y = np.concatenate([
    np.zeros(len(X_quality_dip), dtype=int),
    np.ones(len(X_test_quality_dip), dtype=int)
])

X_domain_train, X_domain_holdout, d_train, d_holdout = train_test_split(
    X_domain,
    domain_y,
    test_size=0.20,
    random_state=42,
    stratify=domain_y
)

domain_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)
domain_model.fit(X_domain_train, d_train)

domain_auc = roc_auc_score(
    d_holdout,
    domain_model.predict_proba(X_domain_holdout)[:, 1]
)
print(f"Domain classifier holdout ROC-AUC: {domain_auc:.4f}")

# Estimate test-likeness for every training row.
p_test_like = domain_model.predict_proba(X_quality_dip)[:, 1]

# Convert probabilities to density-ratio-style weights.
# The class-prior correction is needed because train/test set sizes differ.
n_train = len(X_quality_dip)
n_test = len(X_test_quality_dip)

weights = (
    p_test_like / np.clip(1 - p_test_like, 1e-4, None)
) * (n_train / n_test)

# Cap extreme weights to reduce variance.
weights = np.clip(weights, 0.25, 4.0)
weights = weights / weights.mean()

print(f"Weight range: {weights.min():.3f} to {weights.max():.3f}")
print(f"Weight mean: {weights.mean():.3f}")

Domain classifier holdout ROC-AUC: 0.8304
Weight range: 0.161 to 2.580
Weight mean: 1.000


c:\Dev\Projects\planet-hunter\.venv\Lib\site-packages\sklearn\utils\validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [52]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

# Convert features to NumPy so integer arrays select rows.
X_np = np.asarray(X_quality_dip)
y_np = np.asarray(y)

# Reproduce the exact seed-42 stratified split.
train_idx, val_idx = train_test_split(
    np.arange(len(y_np)),
    test_size=0.20,
    random_state=42,
    stratify=y_np
)

X_train_exp = X_np[train_idx]
X_val_exp = X_np[val_idx]
y_train_exp = y_np[train_idx]
y_val_exp = y_np[val_idx]

# Unweighted baseline
model_unweighted = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0
)
model_unweighted.fit(X_train_exp, y_train_exp)

# Domain-weighted candidate
model_weighted = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0
)
model_weighted.fit(
    X_train_exp,
    y_train_exp,
    sample_weight=weights[train_idx]
)

pred_unweighted = model_unweighted.predict(X_val_exp)
pred_weighted = model_weighted.predict(X_val_exp)

score_unweighted = f1_score(
    y_val_exp, pred_unweighted, average="macro"
)
score_weighted = f1_score(
    y_val_exp, pred_weighted, average="macro"
)

print(f"Unweighted macro-F1: {score_unweighted:.4f}")
print(f"Weighted macro-F1:   {score_weighted:.4f}")
print(f"Difference:          {score_weighted - score_unweighted:+.4f}")

Unweighted macro-F1: 0.5308
Weighted macro-F1:   0.5075
Difference:          -0.0233


In [53]:
import numpy as np
import pandas as pd

X_train_np = np.asarray(X_quality_dip, dtype=np.float64)
X_test_np = np.asarray(X_test_quality_dip, dtype=np.float64)

feature_names = [
    f"feature_{i}" for i in range(X_train_np.shape[1])
]

train_mean = X_train_np.mean(axis=0)
test_mean = X_test_np.mean(axis=0)
train_std = X_train_np.std(axis=0)
test_std = X_test_np.std(axis=0)

eps = 1e-8
mean_shift = np.abs(test_mean - train_mean) / np.maximum(train_std, eps)
std_ratio = test_std / np.maximum(train_std, eps)

shift_report = pd.DataFrame({
    "feature": feature_names,
    "train_mean": train_mean,
    "test_mean": test_mean,
    "mean_shift_in_train_std": mean_shift,
    "train_std": train_std,
    "test_std": test_std,
    "test_train_std_ratio": std_ratio,
})

shift_report = shift_report.sort_values(
    "mean_shift_in_train_std", ascending=False
)

print(shift_report.head(15).round(3).to_string(index=False))

   feature  train_mean  test_mean  mean_shift_in_train_std  train_std  test_std  test_train_std_ratio
feature_21       2.925      3.403                    0.373      1.284     1.872                 1.459
feature_20       4.211      4.868                    0.307      2.138     2.865                 1.340
feature_28       0.425      0.514                    0.265      0.338     0.314                 0.927
feature_23       8.202      9.408                    0.238      5.078     6.630                 1.306
feature_31       0.181      0.235                    0.201      0.265     0.244                 0.918
 feature_5      30.690     22.054                    0.200     43.073    32.508                 0.755
feature_25       3.553      3.857                    0.185      1.642     1.604                 0.977
feature_30       0.287      0.233                    0.173      0.313     0.247                 0.790
feature_29       0.349      0.393                    0.130      0.341     0.297   

In [54]:
# Inspect the feature-extraction code and available feature-name variables.
print("Potential feature-name variables:")
for name, value in globals().copy().items():
    if "feature" in name.lower() and (
        isinstance(value, (list, tuple, pd.Index, np.ndarray))
    ):
        try:
            if len(value) == X_quality_dip.shape[1]:
                print(name, "->", list(value))
        except TypeError:
            pass

print("\nDataFrame columns, if available:")
if hasattr(X_quality_dip, "columns"):
    print(list(X_quality_dip.columns))
else:
    print("X_quality_dip is a NumPy array; inspect feature assembly cells.")

Potential feature-name variables:
feature_names -> ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39']

DataFrame columns, if available:
['std', 'mad', 'min', 'max', 'skew', 'kurtosis', 'p01', 'p05', 'p25', 'p75', 'p95', 'p99', 'tmag', 'teff', 'logg', 'radius', 'contamination', 'trend_slope', 'diff_abs_median', 'diff_std', 'centroid_x_std', 'centroid_x_mad', 'centroid_x_diff_median', 'centroid_x_dev_p95', 'centroid_y_std', 'centroid_y_mad', 'centroid_y_diff_median', 'centroid_y_dev_p95', 'autocor

In [55]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import numpy as np

X = np.asarray(X_quality_dip)
y_arr = np.asarray(y)

train_idx, val_idx = train_test_split(
    np.arange(len(y_arr)),
    test_size=0.20,
    random_state=42,
    stratify=y_arr
)

# Centroid features occupy columns 20–27.
keep_cols = [i for i in range(X.shape[1]) if i not in range(20, 28)]

X_train_full, X_val_full = X[train_idx], X[val_idx]
X_train_no_centroid = X[train_idx][:, keep_cols]
X_val_no_centroid = X[val_idx][:, keep_cols]
y_train, y_val = y_arr[train_idx], y_arr[val_idx]

def fit_and_score(Xtr, Xv):
    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.06,
        class_weight="balanced",
        random_state=0
    )
    model.fit(Xtr, y_train)
    pred = model.predict(Xv)
    return f1_score(y_val, pred, average="macro"), model

baseline_score, _ = fit_and_score(X_train_full, X_val_full)
no_centroid_score, no_centroid_model = fit_and_score(
    X_train_no_centroid, X_val_no_centroid
)

print(f"All 40 features:       {baseline_score:.4f}")
print(f"Without centroid:      {no_centroid_score:.4f}")
print(f"Difference:            {no_centroid_score - baseline_score:+.4f}")

All 40 features:       0.5308
Without centroid:      0.5145
Difference:            -0.0163


In [56]:
# Inspect existing feature matrices and their shapes.
for name in [
    "X_test_quality",
    "X_test_quality_base",
    "X_test_quality_temporal",
    "X_test_quality_centroid",
    "X_test_quality_periodic",
    "X_test_quality_artifacts",
    "X_test_dip",
]:
    if name in globals():
        value = globals()[name]
        print(name, value.shape)

X_test_dip (7000, 4)


In [57]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import numpy as np

X = np.asarray(X_quality_dip)
y_arr = np.asarray(y)

train_idx, val_idx = train_test_split(
    np.arange(len(y_arr)),
    test_size=0.20,
    random_state=42,
    stratify=y_arr
)

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y_arr[train_idx], y_arr[val_idx]

et_model = ExtraTreesClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    max_features=0.9,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

et_model.fit(X_train, y_train)
et_pred = et_model.predict(X_val)

et_score = f1_score(y_val, et_pred, average="macro")

print(f"HistGradientBoosting baseline: 0.5308")
print(f"ExtraTrees validation Macro-F1: {et_score:.4f}")

HistGradientBoosting baseline: 0.5308
ExtraTrees validation Macro-F1: 0.5091


In [ ]:
# Inspect existing feature matrices and their shapes.
for name in [
    "X_test_quality",
    "X_test_quality_base",
    "X_test_quality_temporal",
    "X_test_quality_centroid",
    "X_test_quality_periodic",
    "X_test_quality_artifacts",
    "X_test_dip",
]:
    if name in globals():
        value = globals()[name]
        print(name, value.shape)

X_test_dip (7000, 4)


In [58]:
from sklearn.metrics import classification_report
import pandas as pd

print("BASELINE — HistGradientBoosting")
print(classification_report(y_val, pred_unweighted, digits=3, zero_division=0))

print("\nEXTRATREES")
print(classification_report(y_val, et_pred, digits=3, zero_division=0))

BASELINE — HistGradientBoosting
                   precision    recall  f1-score   support

background_binary      0.459     0.467     0.463       321
 eclipsing_binary      0.825     0.579     0.681       449
       flare_star      0.544     0.499     0.520       361
instrument_glitch      0.650     0.599     0.623       344
           planet      0.411     0.382     0.396       429
   pulsating_star      0.604     0.638     0.621       395
       quiet_star      0.475     0.648     0.548       725
        starspots      0.422     0.370     0.394       576

         accuracy                          0.526      3600
        macro avg      0.549     0.523     0.531      3600
     weighted avg      0.539     0.526     0.527      3600


EXTRATREES
                   precision    recall  f1-score   support

background_binary      0.515     0.386     0.441       321
 eclipsing_binary      0.811     0.563     0.665       449
       flare_star      0.561     0.435     0.490       361
instrume

In [59]:
from sklearn.metrics import f1_score
import numpy as np

# Ensure class columns are aligned before blending.
hgb_classes = model_unweighted.classes_
et_classes = et_model.classes_

assert np.array_equal(hgb_classes, et_classes), (
    "Class orders differ; align probability columns before blending."
)

hgb_probs = model_unweighted.predict_proba(X_val)
et_probs = et_model.predict_proba(X_val)

blend_results = []

for et_weight in [0.05, 0.10, 0.15, 0.20, 0.25]:
    blended_probs = (
        (1 - et_weight) * hgb_probs
        + et_weight * et_probs
    )
    blended_pred = hgb_classes[np.argmax(blended_probs, axis=1)]
    score = f1_score(y_val, blended_pred, average="macro")
    blend_results.append((et_weight, score))

for weight, score in blend_results:
    print(f"ExtraTrees weight={weight:.2f}: Macro-F1={score:.4f}")

print("\nHGB alone: 0.5308")

ExtraTrees weight=0.05: Macro-F1=0.5304
ExtraTrees weight=0.10: Macro-F1=0.5300
ExtraTrees weight=0.15: Macro-F1=0.5314
ExtraTrees weight=0.20: Macro-F1=0.5313
ExtraTrees weight=0.25: Macro-F1=0.5339

HGB alone: 0.5308


In [60]:
from sklearn.metrics import classification_report
import numpy as np

best_et_weight = 0.25

best_blend_probs = (
    (1 - best_et_weight) * hgb_probs
    + best_et_weight * et_probs
)

best_blend_pred = hgb_classes[
    np.argmax(best_blend_probs, axis=1)
]

print("75% HGB + 25% ExtraTrees")
print(classification_report(
    y_val,
    best_blend_pred,
    digits=3,
    zero_division=0
))

print("\nHGB baseline")
print(classification_report(
    y_val,
    pred_unweighted,
    digits=3,
    zero_division=0
))

75% HGB + 25% ExtraTrees
                   precision    recall  f1-score   support

background_binary      0.482     0.464     0.473       321
 eclipsing_binary      0.840     0.575     0.683       449
       flare_star      0.556     0.485     0.518       361
instrument_glitch      0.660     0.593     0.625       344
           planet      0.411     0.378     0.394       429
   pulsating_star      0.600     0.643     0.621       395
       quiet_star      0.470     0.687     0.558       725
        starspots      0.439     0.368     0.400       576

         accuracy                          0.531      3600
        macro avg      0.557     0.524     0.534      3600
     weighted avg      0.546     0.531     0.530      3600


HGB baseline
                   precision    recall  f1-score   support

background_binary      0.459     0.467     0.463       321
 eclipsing_binary      0.825     0.579     0.681       449
       flare_star      0.544     0.499     0.520       361
instrument_gl

In [61]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier

X_full = np.asarray(X_quality_dip)
X_test_full = np.asarray(X_test_quality_dip)
y_full = np.asarray(y)

# Train HGB on all labeled data.
hgb_final = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0
)
hgb_final.fit(X_full, y_full)

# Train ExtraTrees on all labeled data.
et_final = ExtraTreesClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    max_features=0.9,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)
et_final.fit(X_full, y_full)

# Verify the class order before combining probabilities.
assert np.array_equal(hgb_final.classes_, et_final.classes_)

# Blend probabilities: 75% HGB, 25% ExtraTrees.
test_probs = (
    0.75 * hgb_final.predict_proba(X_test_full)
    + 0.25 * et_final.predict_proba(X_test_full)
)

blend_predictions = hgb_final.classes_[np.argmax(test_probs, axis=1)]

# Preserve the competition's sample submission format.
project_dir = Path.cwd().parent
raw_dir = project_dir / "data" / "raw"
sample = pd.read_csv(raw_dir / "sample_submission.csv")

assert len(blend_predictions) == len(sample) == 7000
assert sample.columns.tolist() == ["id", "label"]

submission_blend = sample.copy()
submission_blend["label"] = blend_predictions

output_path = Path.cwd() / "artifacts" / "submission_4_blend.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
submission_blend.to_csv(output_path, index=False)

# Verify the saved file.
check = pd.read_csv(output_path)
assert check.shape == (7000, 2)
assert check["label"].notna().all()

print("Saved:", output_path.resolve())
print("\nPrediction distribution:")
print(check["label"].value_counts())
print("\nFirst five rows:")
print(check.head())

Saved: C:\Dev\Projects\planet-hunter\notebooks\artifacts\submission_4_blend.csv

Prediction distribution:
label
instrument_glitch    2401
starspots             868
pulsating_star        832
flare_star            704
planet                686
eclipsing_binary      574
quiet_star            486
background_binary     449
Name: count, dtype: int64

First five rows:
             id              label
0  TIC914436805         quiet_star
1  TIC964940782  background_binary
2  TIC906185835         quiet_star
3  TIC927004741  instrument_glitch
4  TIC949385375  background_binary
